In [2]:
## Set jax memory preallocation
import os
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.4'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import shutil
import argparse
# import jax
# import jax.numpy as jnp
# from colabdesign.af import mk_af_model
# from colabdesign.af import clear_mem as af_clear_mem
# import colabdesign
# from colabdesign.mpnn import mk_mpnn_model
# from colabdesign.mpnn import clear_mem as mpnn_clear_mem
# from colabdesign.shared.protein import pdb_to_string

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
torch.use_deterministic_algorithms(True, warn_only=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
np.set_printoptions(precision=2)
from scipy.special import softmax
from Bio.PDB import PDBParser

import warnings
from tqdm import tqdm, TqdmExperimentalWarning
import tqdm.notebook
import json
import pickle
import ast
import math

TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'

warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from functools import partialmethod

tqdm.__init__ = partialmethod(tqdm.__init__, leave=False)

from chroma import api
## Register for chroma weights
api.register_key("10e48bde5ef449e3bcee003bf12d5b59")
from chroma import Chroma, Protein, conditioners


class TemplateConfidenceConditioner(conditioners.Conditioner): 
    def __init__(self, template, confidence, c_scale, device="cuda"):
        super().__init__()
        # Setup the coordinates of a 2D lattice
        t_X, t_C, t_S = template.to_XCS()
        confidence = torch.tensor(confidence)[None,:,None,None]
        t_dist = pair_wise_distance_ca(t_X).to(device)
        self.register_buffer("t_X", t_X)
        self.register_buffer("confidence",confidence)
        self.register_buffer("t_dist", t_dist)
        self.c_scale = c_scale

        
    def forward(self, X, C, O, U, t): 
        # Modify potential U based on mean squared difference in pairwise distance
        X_dist = pair_wise_distance_ca(X)
        epsilon = 1e-3
        scaled_loss = (self.c_scale * (self.confidence * (X_dist - self.t_dist)).square()+epsilon).mean(dim=[2,3]).sqrt()
        torch.nn.utils.clip_grad_norm_(scaled_loss, max_norm=3, norm_type=2)
        X.register_hook(lambda g: print(g))
        U_out = U + scaled_loss
        return X, C, O, U_out, t

def pair_wise_distance_ca(X):
    """Assume X has shape of (batches, residues, atoms, coordinates) and alpha-Carbon is at index 1"""
    assert len(X.shape) == 4
    ca = X[:,:,1]
    dist = ((ca[:,:,None] - ca[:,None,:]).square().sum(-1) + 1e-8).sqrt()
    return dist

def entropy(C, dim=-1):
    """By default assumes probability on the last dimension"""
    entropy = -(C * C.log()).sum(dim=dim)
    return entropy

def mpnn_pssm(sequence, pdb_path, mpnn_model, unconditional=False, verbose=False):
    mpnn_model.prep_inputs(pdb_filename=pdb_path, chain="A", verbose=verbose)
    logits = mpnn_model.get_logits()
    pssm = softmax(logits, -1) # Has shape (residues, 20)
    return torch.Tensor(pssm)

def pssm_ce(sequence, pssm):
    log_pssm = pssm.log() # Has shape (residues, 20)
    log_prob_lst = torch.Tensor([res[sequence[n]] for n, res in enumerate(log_pssm)])
    return -log_prob_lst

def rename_folder_pdb(file_dir):
    file_dir_lst = os.listdir(file_dir)
    for file in file_dir_lst:
        new_file_path = "/".join([file_dir, file])
        new_file_path = new_file_path.replace("cycle", "exp")
        os.rename("/".join([file_dir, file]), new_file_path)
        # if file.endswith(".pdb"):
        #     file_path_lst = file.split("_")
        #     suffix = file_path_lst[-1]
        #     if suffix != "init.pdb":
        #         file_path_lst[-1] = file_path_lst[-1][:-4].zfill(3) + ".pdb"
        #     new_file_path = "/".join([file_dir, "_".join(file_path_lst)])
        #     os.rename("/".join([file_dir, file]), new_file_path)

class BestTracker():
    def __init__(self, name, exp_num=None, iter=None, dist_diff=float("inf"), mon_plddt=0, mul_plddt=0, mpnn_ce=float("inf"), mpnn_ent=float("inf"), fname=None, by="multimer"):
        if not by in {"monomer_plddt", "multimer_plddt", "mean_plddt", "dist_diff", "mpnn_ce", "mpnn_ent"}:
            raise Exception("Not a valid choice to rank by.")
        self.name = name
        self.exp_num = exp_num
        self.iter = iter
        self.dist_diff = dist_diff
        self.mon_plddt = mon_plddt
        self.mul_plddt = mul_plddt
        self.mpnn_ce = mpnn_ce
        self.mpnn_ent = mpnn_ent
        self.fname = fname
        self.by = by

    def update(self, exp_num, iter, dist_diff, mon_plddt, mul_plddt, mpnn_ce, mpnn_ent, fname):
        if self.by == "monomer_plddt":
            cond = self.mul_plddt < mul_plddt
        elif self.by == "multimer_plddt":
            cond = self.mon_plddt < mon_plddt
        elif self.by == "mean_plddt":
            cond = ((self.mon_plddt + self.mul_plddt) / 2) < ((mon_plddt + mul_plddt) / 2)
        elif self.by == "dist_diff":
            cond = self.dist_diff > dist_diff
        elif self.by == "mpnn_ce":
            cond = self.mpnn_ce > mpnn_ce
        elif self.by == "mpnn_ent":
            cond = self.mpnn_ent > mpnn_ent

        if cond:
            self.exp_num = exp_num
            self.iter = iter
            self.dist_diff = dist_diff
            self.mon_plddt = mon_plddt
            self.mul_plddt = mul_plddt
            self.mpnn_ce = mpnn_ce
            self.mpnn_ent = mpnn_ent
            self.fname = fname
            self.print_best()

    def print_best(self):
        print(f"{self.name} compared by {self.by}, best_exp: {self.exp_num}, best_iter: {self.iter}, dist_diff: {self.dist_diff:.3g}, mon_plddt: {self.mon_plddt:.3g}, mul_plddt:{self.mul_plddt:.3g}, mpnn_ce:{self.mpnn_ce:.3g}, mpnn_ent:{self.mpnn_ent:.3g}")


In [12]:
pdb_id_lst_file = "/home/ubuntu/ProteinEBM/protein_ebm/data/data_lists/test_decoys.txt"
pdb_id_lst = [line.strip() for line in open(pdb_id_lst_file)]
print(pdb_id_lst[:5])
decoy_single_summary = pd.read_csv("/home/ubuntu/data/af2rank_single/af2rank_single_set_combined_tms_cutoff-190828_in_train_length.csv")
decoy_single_summary = decoy_single_summary[decoy_single_summary["natives_frank"].apply(lambda x: x in pdb_id_lst)]
decoy_single_summary = decoy_single_summary[decoy_single_summary["tms_single"] < 0.5]
decoy_single_summary.to_csv("/home/ubuntu/chroma_af/data/test_decoys_tms-single-05.csv", index=False)
with open("/home/ubuntu/chroma_af/data/test_decoys_tms-single-05.txt", "w") as f:
    for pdb_id in decoy_single_summary["natives_frank"]:
        f.write(f"{pdb_id}\n")
print(len(decoy_single_summary))
decoy_single_summary.head()

['1r6j', '2r2z', '2igd', '2iay', '2hhg']
36


,natives_frank,tms_single,tms_msa,natives_rcsb,denovo,TMscore,in_train,length
13,1enh,0.4517,0.9389,1enh_A,4KYZ,0.9530,True,54
17,1fzy,0.2670,0.9685,1fzy_A,5CWB,0.9313,True,149
23,1i2t,0.4204,0.9599,1i2t_A,5CWI,0.9590,True,61
30,1iz6,0.2628,0.9283,1iz6_A,5TPH,0.9593,True,136
31,1jbe,0.3411,0.9808,1jbe_A,5TPJ,0.9386,True,127


In [ ]:
# pdb_id_lst = ["1AAJ","1ACF","1BK2","1BKR","1CEI","1ENH","1I2T","2B29"] # Try hard ones
# pdb_id_lst = ["1PRQ","1T3X","1XMT","1OPD", "2WWE"] # Try intermediate ones
# pdb_id_lst = ["1ACF","1UI1","1R77","1POH","2I4S"] # Try easy ones
# pdb_id_lst = [("5JYT","A"),("4KSO","A")] # Two conformations of KaiB
pdb_id_lst_file = "/home/ubuntu/ProteinEBM/protein_ebm/data/data_lists/test_decoys.txt"
pdb_id_lst = [line.strip().split() for line in open(pdb_id_lst_file)]
best_plddt_lst = []
best_dist_diff_lst = []
best_mpnn_ce_lst = []
best_mpnn_ent_lst = []
for pdb_id, chain in pdb_id_lst:
    !wget -qnc {f"https://files.rcsb.org/view/{pdb_id}.pdb"}
    best_tracker_plddt = BestTracker(pdb_id, by="mean")
    best_tracker_dist_diff = BestTracker(pdb_id, by="dist_diff")
    best_tracker_mpnn_ce = BestTracker(pdb_id, by="mpnn_ce")
    best_tracker_mpnn_ent = BestTracker(pdb_id, by="mpnn_ent")
    best_plddt_lst.append(best_tracker_plddt)
    best_dist_diff_lst.append(best_tracker_dist_diff)
    best_mpnn_ce_lst.append(best_tracker_mpnn_ce)
    best_mpnn_ent_lst.append(best_tracker_mpnn_ent)
    af_clear_mem()
    mpnn_clear_mem()

    for exp_num in np.arange(20,30,dtype=int):
        # Specify the initial structure; if None, will initialize with random noise using Chroma
        init_fname = None
        num_iter = 30
        print_interval = 10
        plddt_cutoff_schedule = np.linspace(0.6,0.7,num_iter)
        mpnn_ce_cutoff_schedule = np.linspace(-np.log(0.15), -np.log(0.1), num_iter)
        mpnn_ent_cutoff_schedule = np.linspace(-0.15*np.log(0.15) - 0.85*np.log(0.85/19), -0.25*np.log(0.25) - 0.75*np.log(0.75/19), num_iter)
        temperature_schedule = np.linspace(8,8,num_iter)
        noise_schedule = np.linspace(8,0,num_iter)
        alternate = False
        mean_score = True
        tspan = (0.1,0.9)
        gt_fname = f"{pdb_id}.pdb"
        exp_name = f"{pdb_id}_exp_{exp_num:03d}"
        save_dir = f"/home/ubuntu/chroma_af/results/{exp_name}"
        if not os.path.isdir(save_dir):
            os.mkdir(save_dir)
        tem_fname = "/".join([save_dir,f"{exp_name.zfill(3)}_init.pdb"])

        # Set up AF design
        af = mk_af_model(protocol="fixbb", use_templates=True, debug=True)
        af_mul = mk_af_model(protocol="fixbb", use_templates=True, debug=True, model_type="alphafold2_multimer_v3")
        af.prep_inputs(gt_fname,chain=chain)
        af_mul.prep_inputs(gt_fname,chain=chain)
        gt_seq = af._inputs["batch"]["aatype"]
        num_resi = len(gt_seq)

        # Set up ProteinMPNN
        mpnn = mk_mpnn_model()

        # Set up Chroma
        chroma = Chroma(device=device)

        # Set up initial structure
        if init_fname is not None:
            protein_init = shutil.copy2(init_fname, tem_fname)
        if not os.path.exists(tem_fname):
            protein_init = chroma.sample(chain_lengths=[num_resi], design_method=None)
            protein_init.to(tem_fname)

        # Setup "distogram" for ground truth structure
        gt_dist = pair_wise_distance_ca(Protein.from_PDB(gt_fname).to_XCS()[0][:,:num_resi])
        dist_diff_lst = np.empty(num_iter)
        plddt_lst = np.empty(num_iter)
        pae_lst = np.empty(num_iter)
        cmap_ent_lst = np.empty(num_iter)
        mean_mpnn_ce_lst = np.empty(num_iter)
        mean_mpnn_ent_lst = np.empty(num_iter)
        
        for i in range(num_iter):
            af.prep_inputs(tem_fname, chain=chain)
            af.predict(gt_seq)
            af_mul.prep_inputs(tem_fname, chain=chain)
            af_mul.predict(gt_seq)
            
            # Alternate between AF monomer and multimer models to avoid adversarial samples
            cur_af = af if (not alternate) or (i%2)==0 else af_mul
            tem_fname = "/".join([save_dir,f"{exp_name}_af_{i:03d}.pdb"])
            cur_af.save_pdb(tem_fname)

            # Calculate difference in pair-wise distance
            af_dist = pair_wise_distance_ca(Protein.from_PDB(tem_fname).to_XCS()[0])
            mean_dist_diff = (af_dist - gt_dist).abs().mean()
            dist_diff_lst[i] = mean_dist_diff

            # Calculate pLDDT
            mon_plddt = torch.tensor(af.aux["plddt"])
            mul_plddt = torch.tensor(af_mul.aux["plddt"])
            plddt = torch.stack([mon_plddt, mul_plddt], dim=0).mean(dim=0) if mean_score else torch.tensor(cur_af.aux["plddt"])
            mean_plddt = plddt.mean()
            plddt_lst[i] = mean_plddt

            # Calculate pAE
            pae = torch.tensor(cur_af.aux["pae"]).to(device).mean(1) / 32 # Scale pAE to (0,1) and take average by dim1
            mean_pae = torch.mean(pae)
            pae_lst[i] = mean_pae

            # Calculate contact map
            cmap = torch.tensor(cur_af.aux["cmap"]).to(device)
            cmap_ent = entropy(torch.softmax(cmap,dim=-1)) / (np.log(num_resi) * num_resi) # Normalize
            mean_cmap_ent = cmap_ent.mean()
            cmap_ent_lst[i] = mean_cmap_ent

            # Calculate mpnn cross-entropy
            mpnn_prob = mpnn_pssm(gt_seq, tem_fname, mpnn)
            mpnn_ent = entropy(mpnn_prob, dim=1)
            mpnn_ce = pssm_ce(gt_seq, mpnn_prob)
            mean_mpnn_ce = mpnn_ce.mean()
            mean_mpnn_ce_lst[i] = mean_mpnn_ce
            mean_mpnn_ent = mpnn_ent.mean()
            mean_mpnn_ent_lst[i] = mean_mpnn_ent

            # Print metrics
            print(f"Iteration {i}: dist_diff={mean_dist_diff:.3g}, plddt={mean_plddt:.3g}, pae={mean_pae:.3g}, cmap_ent={mean_cmap_ent:.3g}, mpnn_ce={mean_mpnn_ce:.3g}, mpnn_ent={mean_mpnn_ent:.3g}")

            # Update best structure so far
            mean_mon_plddt = mon_plddt.mean()
            mean_mul_plddt = mul_plddt.mean()
            best_tracker_plddt.update(exp_num, i, mean_dist_diff, mean_mon_plddt, mean_mul_plddt, mean_mpnn_ce, mean_mpnn_ent, tem_fname)
            best_tracker_dist_diff.update(exp_num, i, mean_dist_diff, mean_mon_plddt, mean_mul_plddt, mean_mpnn_ce, mean_mpnn_ent, tem_fname)
            best_tracker_mpnn_ce.update(exp_num, i, mean_dist_diff, mean_mon_plddt, mean_mul_plddt, mean_mpnn_ce, mean_mpnn_ent, tem_fname)
            best_tracker_mpnn_ent.update(exp_num, i, mean_dist_diff, mean_mon_plddt, mean_mul_plddt, mean_mpnn_ce, mean_mpnn_ent, tem_fname)

            # Create mask
            # score = plddt[None,:,None,None].expand(-1, -1, 4, 3) # To match the shape of X
            plddt_cutoff = plddt_cutoff_schedule[i]
            mpnn_ce_cutoff = mpnn_ce_cutoff_schedule[i]
            mpnn_ent_cutoff = mpnn_ent_cutoff_schedule[i]

            mask = None
            # Use pLDDT for masking
            if plddt.max() > plddt_cutoff:
                plddt_cutoff = min(plddt_cutoff, torch.quantile(plddt, 0.25, interpolation='linear'))
                mask = plddt > plddt_cutoff

            # Use ProteinMPNN predicted sequence cross-entropy for masking
            # if mpnn_ce.min() < mpnn_ce_cutoff:
            #     mpnn_ce_cutoff = min(mpnn_ce_cutoff, torch.quantile(mpnn_ce, 0.75, interpolation='linear'))
            #     mask = mpnn_ce < mpnn_ce_cutoff

            # Use ProteinMPNN predicted sequence entropy for masking
            # if mpnn_ent.min() < mpnn_ent_cutoff:
            #     mpnn_ent_cutoff = min(mpnn_ent_cutoff, torch.quantile(mpnn_ent, 0.75, interpolation='linear'))
            #     mask = mpnn_ent < mpnn_ent_cutoff

            expression = "all" # This is not used if selection_indices is provided

            protein_af = Protein.from_PDB(tem_fname)
            substructure_conditioner = conditioners.SubstructureConditioner(protein_af,
                                                                            backbone_model=chroma.backbone_network,
                                                                            selection=expression,
                                                                            selection_indices=mask,
                                                                            tspan=tspan,
                                                                           ).to(device)

            temperature = temperature_schedule[i]
            noise = noise_schedule[i]
            protein_ch = chroma.sample(chain_lengths=[num_resi],
                                    initialize_noise=True,
                                    conditioner=substructure_conditioner,
                                    langevin_factor=noise,
                                    langevin_isothermal=True,
                                    inverse_temperature=temperature,
                                    sde_func='langevin',
                                    steps=400,
                                    design_method=None,
                                    # full_output=True,
                                    # trajectory_length=500,
            )
            tem_fname = "/".join([save_dir,f"{exp_name}_ch_{i:03d}.pdb"])
            protein_ch.to(tem_fname)

            # Debug
            ch_dist = pair_wise_distance_ca(Protein.from_PDB(tem_fname).to_XCS()[0])
            af_ch_dist_diff = (ch_dist - af_dist).abs().mean()
            print(f"dist_diff between AF and Chroma structure: {af_ch_dist_diff}")

            if (i+1) % print_interval == 0:
                plt.plot(range(i+1), dist_diff_lst[:i+1])
                plt.title(f"{exp_name} pair-wise distance difference")
                plt.savefig("/".join([save_dir, f"{exp_name}_dist_diff_fig.jpg"]))
                plt.show()
                plt.clf()
                
                plt.plot(range(i+1), plddt_lst[:i+1])
                plt.title(f"{exp_name} pLDDT")
                plt.savefig("/".join([save_dir, f"{exp_name}_plddt_fig.jpg"]))
                plt.show()
                plt.clf()
                
                plt.plot(range(i+1), pae_lst[:i+1])
                plt.title(f"{exp_name} pAE")
                plt.savefig("/".join([save_dir, f"{exp_name}_pae_fig.jpg"]))
                plt.show()
                plt.clf()

                plt.plot(range(i+1), mean_mpnn_ce_lst[:i+1])
                plt.title(f"{exp_name} ProteinMPNN Cross-Entropy")
                plt.savefig("/".join([save_dir, f"{exp_name}_mpnn_ce_fig.jpg"]))
                plt.show()
                plt.clf()

                plt.plot(range(i+1), mean_mpnn_ent_lst[:i+1])
                plt.title(f"{exp_name} ProteinMPNN Entropy")
                plt.savefig("/".join([save_dir, f"{exp_name}_mpnn_ent_fig.jpg"]))
                plt.show()
                plt.clf()
                
                plt.plot(range(i+1), cmap_ent_lst[:i+1])
                plt.title(f"{exp_name} CMap Mean Entropy")
                plt.savefig("/".join([save_dir, f"{exp_name}_cmap_ent_fig.jpg"]))
                plt.show()
                plt.clf()

    best_tracker_plddt.print_best()
    best_tracker_dist_diff.print_best()
    best_tracker_mpnn_ce.print_best()
    best_tracker_mpnn_ent.print_best()

In [ ]:
[best_tracker.print_best() for best_tracker in best_plddt_lst]
[best_tracker.print_best() for best_tracker in best_dist_diff_lst]
[best_tracker.print_best() for best_tracker in best_mpnn_ce_lst]
[best_tracker.print_best() for best_tracker in best_mpnn_ent_lst]

In [ ]:
mpnn = mk_mpnn_model()
af = mk_af_model(protocol="fixbb", use_templates=True, debug=True)
af.prep_inputs("1CC8.pdb",chain="A")
gt_seq = af._inputs["batch"]["aatype"]
gt_mpnn_prob = mpnn_pssm(gt_seq, "1CC8.pdb", mpnn)
af_mpnn_prob = mpnn_pssm(gt_seq, "1CC8_exp_079/1CC8_exp_79_af_029.pdb", mpnn)
gt_mpnn_ce = pssm_ce(gt_seq, gt_mpnn_prob)
af_mpnn_ce = pssm_ce(gt_seq, af_mpnn_prob)
print(gt_mpnn_ce.mean(), af_mpnn_ce.mean())


plt.title("Difference in Target Residue Predicted Probability")
plt.xlabel("position of amino acid")
plt.ylabel("CE difference (experimental-AF)")
plt.imshow(np.diag((-gt_mpnn_ce).exp() - (-af_mpnn_ce).exp()))
plt.colorbar()
plt.savefig("1CC8_exp_79_mpnn_target_prob_comparison")
plt.show()

plt.title("Difference in Max Predicted Probability")
plt.xlabel("position of amino acid")
plt.ylabel("probability difference (experimental-AF)")
plt.imshow(np.diag(torch.max(gt_mpnn_prob, axis=1)[0] - torch.max(af_mpnn_prob, axis=1)[0]))
plt.colorbar()
plt.savefig("1CC8_exp_79_mpnn_max_prob_comparison")
plt.show()

In [ ]:
# Sanity check for consistency of amino acid code idnexing, using PDB_ID=1CC8

amino_acids = ["ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS",
                   "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP",
                   "TYR", "VAL"]
aa_mapping = {
        "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
        "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
        "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
        "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V"
    }

"AEIKHYQFNVVMTCSGCSGAVNKVLTKLEPDVSKIDISLEKQLVDVYTTLPYDFILEKIKKTGKEVRSGKQL" == "".join([aa_mapping[amino_acids[aa]] for aa in gt_seq])

In [ ]:
import itertools

for pdb_id in ["1CC8"]:
    !wget -qnc {f"https://files.rcsb.org/view/{pdb_id}.pdb"}
    cut_sch_lst = [np.linspace(0.6,0.6,num_iter), np.linspace(0.7,0.6,num_iter), np.linspace(0.75,0.6,num_iter)]
    tem_sch_lst = [np.linspace(8,8,num_iter), np.linspace(10,6,num_iter), np.linspace(12,4,num_iter)]
    noi_sch_lst = [np.linspace(4,4,num_iter), np.linspace(6,2,num_iter), np.linspace(8,0,num_iter)]
    cycle_num = 79
    best_dist_diff = float("inf")
    best_plddt = 0
    combs = [[0,1,2] for _ in range(3)]
    for plddt_cutoff_schedule_i, temperature_schedule_i, noise_schedule_i in list(itertools.product(*combs)):
        num_iter = 30
        print_int = 5
        scale = 5
        init_af = False
        rg = False

        plddt_cutoff_schedule = cut_sch_lst[plddt_cutoff_schedule_i]
        temperature_schedule = tem_sch_lst[temperature_schedule_i]
        noise_schedule = noi_sch_lst[noise_schedule_i]
        
        tspan = (0.1,0.9)
        gt_fname = f"{pdb_id}.pdb"
        cycle_name = f"{pdb_id}_cycle_{int(cycle_num)}_c{plddt_cutoff_schedule_i}_t{temperature_schedule_i}_n{noise_schedule_i}"
        save_dir = f"/home/ubuntu/{cycle_name}"
        if not os.path.isdir(save_dir):
            os.mkdir(save_dir)
        tem_fname = "/".join([save_dir,f"{pdb_id}_cycle_{int(cycle_num)}_init.pdb"])
        
        # Setup AF design
        af = mk_af_model(protocol="fixbb", use_templates=True, debug=True)
        af.prep_inputs(gt_fname,chain="A")
        gt_seq = af._inputs["batch"]["aatype"]
        num_resi = len(gt_seq)
        
        # Setup Chroma
        chroma = Chroma(device=device)
        if not os.path.exists(tem_fname):
            if init_af:
                af_ = mk_af_model(protocol="fixbb", use_templates=False, debug=True)
                af_.prep_inputs(gt_fname,chain="A")
                af_.predict(gt_seq)
                af_.save_pdb(tem_fname)
            else:
                protein_diff = chroma.sample(chain_lengths=[num_resi], design_method=None)
                protein_diff.to(tem_fname)

        # Setup "distogram" for ground truth structure
        gt_dist = pair_wise_distance_ca(Protein.from_PDB(gt_fname).to_XCS()[0][:,:num_resi])
        dist_diff_lst = np.empty(num_iter)
        plddt_lst = np.empty(num_iter)
        pae_lst = np.empty(num_iter)
        cmap_ent_lst = np.empty(num_iter)
        
        for i in range(num_iter):
            af.prep_inputs(tem_fname, chain="A")
            af.predict(gt_seq)
            tem_fname = "/".join([save_dir,f"{cycle_name}_af_{i}.pdb"])
            af.save_pdb(tem_fname)
        
            # Calculate difference in pair-wise distance
            af_dist = pair_wise_distance_ca(Protein.from_PDB(tem_fname).to_XCS()[0])
            mean_dist_diff = (af_dist - gt_dist).abs().mean()
            dist_diff_lst[i] = mean_dist_diff
            
            plddt = torch.tensor(af.aux["plddt"]).to(device)
            mean_plddt = torch.mean(plddt)
            plddt_lst[i] = mean_plddt
        
            pae = torch.tensor(af.aux["pae"]).to(device).mean(1) / 32 # Scale pAE to (0,1) and take average by dim1
            mean_pae = torch.mean(pae)
            pae_lst[i] = mean_pae
        
            cmap = torch.tensor(af.aux["cmap"]).to(device)
            cmap_ent = entropy(torch.softmax(cmap,dim=-1)) / (np.log(num_resi) * num_resi) # Normalize
            mean_cmap_ent = cmap_ent.mean()
            cmap_ent_lst[i] = mean_cmap_ent
            print(f"Iteration {i}: dist_diff={mean_dist_diff:.3f}, plddt={mean_plddt:.3f}, pae={mean_pae:.3f}, cmap_ent={mean_cmap_ent:.3f}")

            # Update best structure so far
            if mean_dist_diff < best_dist_diff:
                best_dist_diff = dist_diff
                best_plddt = mean_plddt
                best_c = plddt_cutoff_schedule_i
                best_t = temperature_schedule_i
                best_n = noise_schedule_i
                print(f"New best! best_ctn:{best_c},{best_t},{best_n}, dist_diff: {best_dist_diff}, plddt: {best_plddt}")

            # Create mask
            score = plddt
            # score = score[None,:,None,None].expand(-1, -1, 4, 3) # To match the shape of X
            plddt_cutoff = plddt_cutoff_schedule[i]
            if plddt.max() > plddt_cutoff:
                plddt_cutoff = min(plddt_cutoff, torch.quantile(score, 0.25, interpolation='linear'))
                mask = score > plddt_cutoff
            else:
                mask = None
            print(mask)
            expression = "all" # This is not used if selection_indices is provided
        
            protein_af = Protein.from_PDB(tem_fname)
            substructure_conditioner = conditioners.SubstructureConditioner(protein_af,
                                                                            backbone_model=chroma.backbone_network,
                                                                            rg=rg,
                                                                            selection=expression,
                                                                            selection_indices=mask,
                                                                            tspan=tspan,
                                                                           ).to(device)

            temperature = temperature_schedule[i]
            noise = noise_schedule[i]
            protein_ch = chroma.sample(chain_lengths=[num_resi],
                                    initialize_noise=True,
                                    conditioner=substructure_conditioner,
                                    langevin_factor=noise,
                                    langevin_isothermal=True,
                                    inverse_temperature=temperature,
                                    sde_func='langevin',
                                    steps=500,
                                    design_method=None,
                                    # full_output=True,
                                    # trajectory_length=500,
            )
            tem_fname = "/".join([save_dir,f"{cycle_name}_ch_{i}.pdb"])
            protein_ch.to(tem_fname)

            # Debug
            ch_dist = pair_wise_distance_ca(Protein.from_PDB(tem_fname).to_XCS()[0])
            af_ch_dist_diff = (ch_dist - af_dist).abs().mean()
            print(f"dist_diff between AF and Chroma structure: {af_ch_dist_diff}")

            if (i+1) % print_int == 0:
                plt.plot(range(i+1), dist_diff_lst[:i+1])
                plt.title(f"{cycle_name} pair-wise distance difference")
                plt.savefig("/".join([save_dir, f"{cycle_name}_dist_diff_fig.jpg"]))
                plt.show()
                plt.clf()
                
                plt.plot(range(i+1), plddt_lst[:i+1])
                plt.title(f"{cycle_name} pLDDT")
                plt.savefig("/".join([save_dir, f"{cycle_name}_plddt_fig.jpg"]))
                plt.show()
                plt.clf()
                
                plt.plot(range(i+1), pae_lst[:i+1])
                plt.title(f"{cycle_name} pAE")
                plt.savefig("/".join([save_dir, f"{cycle_name}_pae_fig.jpg"]))
                plt.show()
                plt.clf()

                plt.plot(range(i+1), cmap_ent_lst[:i+1])
                plt.title(f"{cycle_name} CMap Mean Entropy")
                plt.savefig("/".join([save_dir, f"{cycle_name}_cmap_ent_fig.jpg"]))
                plt.show()
                plt.clf()
                
print(f"Overall best: best_ctn:{best_c},{best_t},{best_n}, dist_diff: {best_dist_diff}, plddt: {best_plddt}")
